In [2]:
#@title MASTER — Persistent OpenVL + FrankaPick + SimplerEnv Setup
!apt-get update -qq
!apt-get install -y -qq python3.12-venv

# ============================================================
# Persistent OpenVLA environment on Google Drive
# Installs only when required versions are missing/different
# ============================================================

import os
import sys
import subprocess
import shutil
import importlib.metadata as metadata

from google.colab import drive

# ------------------------------------------------------------
# 1. Mount Google Drive
# and Persistent Hugging Face cache
# ------------------------------------------------------------

drive.mount("/content/drive")


HF_CACHE = "/content/drive/MyDrive/huggingface_cache"

os.makedirs(HF_CACHE, exist_ok=True)

os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE

print("Hugging Face cache:", HF_CACHE)

ENV_PATH = "/content/drive/MyDrive/colab_env_openvla"
VENV_PYTHON = os.path.join(ENV_PATH, "bin", "python")

'''
if os.path.exists(ENV_PATH):
    print("Removing old/non-venv directory:")
    print(ENV_PATH)
    shutil.rmtree(ENV_PATH)
'''

print("Creating fresh persistent venv...")

# ------------------------------------------------------------
# 2. Versions we require
# ------------------------------------------------------------

REQUIRED = {
    "transformers": "4.40.1",
    "tokenizers": "0.19.1",
    "accelerate": "0.26.0",
    "timm": "0.9.10",
    "peft": "0.11.1",
    "sentencepiece": "0.1.99",
    "draccus": "0.8.0",
}

# ------------------------------------------------------------
# 3. Create persistent venv if necessary
# ------------------------------------------------------------

if not os.path.exists(VENV_PYTHON):

    print("================================================")
    print("Creating persistent OpenVLA environment")
    print("================================================")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "venv",
            "--system-site-packages",
            ENV_PATH,
        ],
        check=True,
    )

    print("✅ Virtual environment created")
else:
    print("✅ Persistent environment already exists")


# ------------------------------------------------------------
# 4. Check installed versions INSIDE the venv
# ------------------------------------------------------------

def get_installed_version(package):
    """
    Ask the persistent venv's Python which version is installed.
    """
    result = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            (
                "import importlib.metadata as m; "
                f"print(m.version('{package}'))"
            ),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        return None

    return result.stdout.strip()


print("\n================================================")
print("Checking OpenVLA environment")
print("================================================")

needs_install = []

for package, required_version in REQUIRED.items():

    installed = get_installed_version(package)

    if installed is None:
        print(f"❌ {package}: NOT INSTALLED")
        needs_install.append(package)

    elif required_version is None:
        print(f"✅ {package}: {installed}")

    elif installed != required_version:
        print(
            f"⚠️ {package}: {installed} "
            f"(required {required_version})"
        )
        needs_install.append(package)

    else:
        print(f"✅ {package}: {installed}")


# ------------------------------------------------------------
# 5. Install ONLY if something is missing/wrong
# ------------------------------------------------------------

if needs_install:

    print("\n================================================")
    print("Installing/updating OpenVLA dependencies")
    print("================================================")

    packages = [
        "transformers==4.40.1",
        "tokenizers==0.19.1",
        "accelerate==0.26.0",
        "timm==0.9.10",
        "peft==0.11.1",
        "sentencepiece==0.1.99",
        "draccus==0.8.0",
    ]

    subprocess.run(
        [
            VENV_PYTHON,
            "-m",
            "pip",
            "install",
            "-q",
            *packages,
        ],
        check=True,
    )

    print("✅ Installation/update complete")

else:

    print("\n================================================")
    print("🎉 OpenVLA environment already satisfies all")
    print("   required versions.")
    print("   SKIPPING pip installation.")
    print("================================================")


# ------------------------------------------------------------
# 6. Make the venv packages available to THIS notebook
# ------------------------------------------------------------

# We are using Colab's Python kernel, so add the venv's
# site-packages to sys.path.

PYTHON_VERSION = f"{sys.version_info.major}.{sys.version_info.minor}"

VENV_SITE = os.path.join(
    ENV_PATH,
    "lib",
    f"python{PYTHON_VERSION}",
    "site-packages",
)

if os.path.exists(VENV_SITE):

    if VENV_SITE not in sys.path:
        sys.path.insert(0, VENV_SITE)

    print("\nVenv site-packages:")
    print(VENV_SITE)

else:

    print("\n⚠️ WARNING:")
    print("Could not find:")
    print(VENV_SITE)


# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

print("\n================================================")
print("FINAL ENVIRONMENT CHECK")
print("================================================")

print("Notebook Python:")
print(sys.version)

print("\nNotebook executable:")
print(sys.executable)


#import transformers
#import tokenizers
#import accelerate
#import timm
#import peft

#print("\nTransformers :", transformers.__version__)
#print("Tokenizers   :", tokenizers.__version__)
#print("Accelerate   :", accelerate.__version__)
#print("Timm         :", timm.__version__)
##print("PEFT         :", peft.__version__)

#print("\nTransformers loaded from:")
#print(transformers.__file__)

print("\n✅ Persistent OpenVLA environment ready.")

!pip install -q mujoco==3.8.0 mediapy

print("mediapy and mujoco installed")

W: An error occurred during the signature verification. The repository is not updated and the previous index files will be used. GPG error: https://cli.github.com/packages stable InRelease: The following signatures couldn't be verified because the public key is not available: NO_PUBKEY 5612B36462313325
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://cli.github.com/packages/dists/stable/InRelease  The following signatures couldn't be verified because the public key is not available: NO_PUBKEY 5612B36462313325
W: Some index files failed to download. They have been ignored, or old ones used instead.
Mounted at /content/drive
Hugging Face cache: /content/drive/MyDrive/huggingface_cache
Creating fresh persistent venv...
✅ Persistent environment already exists

Checking OpenVLA environment
✅ transformers: 4.40.1
✅ tokenizers: 

In [ ]:
#@title inference
import os
# Force GPU-accelerated offscreen rendering
os.environ["MUJOCO_GL"] = "egl"

import mujoco
print("MuJoCo:", mujoco.__version__)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))

import numpy as np
import PIL.Image

import scipy
print("SciPy      :", scipy.__version__, scipy.__file__)

from scipy.spatial.transform import Rotation as R
import time

import mediapy as media
print("Mediapy    :", media.__file__)

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import transformers
print("Transformers:", transformers.__version__)

from transformers import AutoModelForVision2Seq
print("AutoModelForVision2Seq OK")

from transformers import AutoProcessor
print("AutoProcessor OK")

#from transformers import AutoModelForVision2Seq, AutoProcessor

import sys
print("\n--- Environment Health Check ---")
print(f"Active Python:     {sys.version.split()[0]}")
print(f"Active Python Executable: {sys.executable}")

# ==========================================
# CONFIGURATION & CONSTANTS
# ==========================================
TASK_PROMPT = "In: What action should the robot take to pick up the red cube?\nOut:"
UNNORM_KEY = "bridge_orig"  # Change to your dataset domain key if using Libero/custom (e.g., "libero_spatial")

EMA_ALPHA = 1.0
POSITION_SCALE = 0.7
ROTATION_SCALE = 0.02

ARM_JOINT_IDS = list(range(7))      # Joint IDs 0..6
GRIPPER_CTRL_ID = 7               # Actuator index for Franka finger control

class IK:
    def __init__(self, site_id):
        self.site_id = site_id

    def get_position_error(self, data, target_pos, dbg_print=False):
        current_pos = data.site_xpos[self.site_id].copy()
        pos_error = target_pos - current_pos
        if dbg_print:
            print("CURRENT EE:", np.round(current_pos, 5))
            print("TARGET EE:", np.round(target_pos, 5))
            print("EE error:", pos_error, "meters")
        return pos_error

    def get_rotation_error(self, data, target_rot, dbg_print=False):
        current_rot = data.site_xmat[self.site_id].reshape(3,3)
        rot_error_mat = target_rot @ current_rot.T
        return R.from_matrix(rot_error_mat).as_rotvec()

    def calculate(self, model, data, target_pos, target_rot):
        jacp = np.zeros((3, model.nv))
        jacr = np.zeros((3, model.nv))
        mujoco.mj_jacSite(model, data, jacp, jacr, self.site_id)

        J = np.vstack((jacp[:, :7], jacr[:, :7]))
        lam = 0.01
        J_inv = (J.T @ np.linalg.inv(J @ J.T + (lam ** 2) * np.eye(6)))

        pos_error = self.get_position_error(data, target_pos)
        rot_error = self.get_rotation_error(data, target_rot)

        delta_pose = np.concatenate([pos_error, rot_error])

        max_pos_step = 0.07
        pos_norm = np.linalg.norm(delta_pose[:3])
        if pos_norm > max_pos_step:
            delta_pose[:3] *= max_pos_step / pos_norm

        dq = J_inv @ delta_pose
        q_current = data.qpos[:7].copy()
        joint_ranges = model.jnt_range[ARM_JOINT_IDS]

        return np.clip(
            q_current + dq,
            joint_ranges[:, 0],
            joint_ranges[:, 1])


class OpenVLAController:
    def __init__(self):
        self.smoothed_spatial_action = np.zeros(6)
        print("CUDA available:", torch.cuda.is_available())
        self.processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)

        self.vla = AutoModelForVision2Seq.from_pretrained(
            "openvla/openvla-7b",
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            device_map="auto"
        )

        print("Device:", self.vla.device)
        print("Model dtype:", next(self.vla.parameters()).dtype)
        print(self.vla.hf_device_map)
        print("✅ OpenVLA-7B loaded successfully!")


    def predict_action(self, image):
        """Passes image and prompt directly to the loaded OpenVLA model."""
        t0 = time.perf_counter()
        inputs = self.processor(TASK_PROMPT, image)
        t1 = time.perf_counter()
        # Move inputs to GPU
        inputs = inputs.to("cuda:0")
        # Match the OpenVLA vision model's BF16 weights
        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
        # Make sure GPU work before inference is accounted for
        torch.cuda.synchronize()
        t2 = time.perf_counter()
        action = self.vla.predict_action(**inputs, unnorm_key=UNNORM_KEY, do_sample=False)
        '''
        action = np.array(
          [0.012, -0.034, 0.008, 0.001, -0.005, 0.012, 0.98],
          dtype=np.float32
        )
        '''

        torch.cuda.synchronize()
        t3 = time.perf_counter()
        '''
        print(
            f"Processor: {t1-t0:.3f}s | "
            f"GPU transfer: {t2-t1:.3f}s | "
            f"VLA inference: {t3-t2:.3f}s"
        )
        '''
        return action

    def process_raw_action(self, raw_action):
        raw_spatial = raw_action[:6]
        self.smoothed_spatial_action = (
            (1.0 - EMA_ALPHA) * self.smoothed_spatial_action +
            EMA_ALPHA * raw_spatial
        )
        raw_gripper = raw_action[6]
        return self.smoothed_spatial_action, raw_gripper

    def compute_delta_pose(self, smoothed_spatial, R_cam):
        raw_xyz = np.clip(smoothed_spatial[:3], -1.0, 1.0) * POSITION_SCALE
        raw_rpy = np.clip(smoothed_spatial[3:6], -1.0, 1.0) * ROTATION_SCALE
        delta_xyz_world = R_cam.T @ raw_xyz
        delta_rotvec_world = R_cam.T @ raw_rpy
        return delta_xyz_world, delta_rotvec_world


def scale_gripper(model, raw_gripper, actuator_id=7):
    raw = np.clip(raw_gripper, 0.0, 1.0)
    ctrl_min, ctrl_max = model.actuator_ctrlrange[actuator_id]
    return ctrl_min + raw * (ctrl_max - ctrl_min)

def open_gripper(model, data):
    data.ctrl[GRIPPER_CTRL_ID] = scale_gripper(model, 1.0)
    for _ in range(50):
        mujoco.mj_step(model, data)

def close_gripper(model, data):
    data.ctrl[GRIPPER_CTRL_ID] = scale_gripper(model, 0.0)
    for _ in range(50):
        mujoco.mj_step(model, data)

def run_simulation():
    model_path = "drive/MyDrive/franka_emika_panda/vla_task.xml"
    if not os.path.exists(model_path):
        print(f"Error: Model file not found at {model_path}.")
        return

    model = mujoco.MjModel.from_xml_path(model_path)
    print("opened mujoco model")
    data = mujoco.MjData(model)
    mujoco.mj_resetData(model, data)

    # Set bent "ready" pose to break singularity
    ready_qpos = [0.0, -0.785, 0.0, -2.356, 0.0, 1.571, 0.785]
    data.qpos[ARM_JOINT_IDS] = ready_qpos
    mujoco.mj_forward(model, data)

    print("\n===== Sites =====")
    for i in range(model.nsite):
        print(i, mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_SITE, i))

    renderer = mujoco.Renderer(model, height=224, width=224)
    print("Rendered  --- panda is totally fine")

    site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, "pinch")
    if site_id == -1:
        print("Error: 'pinch' site does not exist in model XML.")
        return

    current_pos = data.site_xpos[site_id].copy()
    current_rot = data.site_xmat[site_id].reshape(3,3).copy()
    target_pos = current_pos
    target_rot = current_rot
    data.ctrl[GRIPPER_CTRL_ID] = scale_gripper(model, 1.0)
    mujoco.mj_step(model, data)


    print("\n===== CAMERAS =====")
    for i in range(model.ncam):
        name = mujoco.mj_id2name(
            model,
            mujoco.mjtObj.mjOBJ_CAMERA,
            i
        )

        print(
            i,
            name,
            "pos =", model.cam_pos[i],
            "quat =", model.cam_quat[i],
            "mode =", model.cam_mode[i]
        )

    cam_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, "vla_camera")
    if cam_id == -1:
        print("cam_id is forced to 0")
        cam_id = 0

    print("\nvla_camera ID:", cam_id)

    if cam_id >= 0:
        print("vla_camera pos:", model.cam_pos[cam_id])
        print("vla_camera quat:", model.cam_quat[cam_id])
        print("vla_camera mode:", model.cam_mode[cam_id])

    frames = []
    ik = IK(site_id)
    controller = OpenVLAController()
    max_iterations = 200  # Adjust rollout step length as needed
    print(vars(controller).keys())

    cube_id = model.body("cube").id
    cube_pos = data.xpos[cube_id]
    print("Cube position:", cube_pos)
    print("Hand position:", data.xpos[model.body("hand").id])

    print("🚀 Starting MuJoCo OpenVLA Control Loop...")

    for iteration in range(max_iterations):
        t0 = time.perf_counter()
        # 1. Render camera image for VLA observation
        renderer.update_scene(data, camera=cam_id)
        pixels = renderer.render()

        if iteration % 10 == 0:
          media.show_image(pixels)

        frames.append(pixels)
        img = PIL.Image.fromarray(pixels)

        t1 = time.perf_counter()
        # 2. Get direct action prediction from OpenVLA
        raw_action = controller.predict_action(img)
        t2 = time.perf_counter()
        print(f"Iteration {iteration} - Raw Action: {raw_action}")

        smoothed_spatial, raw_gripper = controller.process_raw_action(raw_action)

        # 3. Compute delta pose
        R_cam = data.cam_xmat[cam_id].reshape(3, 3)
        delta_xyz_world, delta_rotvec_world = controller.compute_delta_pose(smoothed_spatial, R_cam)

        TARGET_STEP = 0.02
        distance = np.linalg.norm(delta_xyz_world)
        if distance > TARGET_STEP:
            delta_xyz_world = (delta_xyz_world / distance) * TARGET_STEP

        current_pos = data.site_xpos[site_id].copy()
        target_pos = current_pos + delta_xyz_world

        t3 = time.perf_counter()
        # 4. Execute Inverse Kinematics steps
        for _ in range(10):
            target_q = ik.calculate(model, data, target_pos, target_rot)
            data.ctrl[:7] = target_q
            for _ in range(10):
                mujoco.mj_step(model, data)

            err = np.linalg.norm(target_pos - data.site_xpos[site_id])
            if err < 0.005:
                break

        t4 = time.perf_counter()

        new_pos = data.site_xpos[site_id].copy()
        print("Actual EE motion:", new_pos - current_pos)

        error = target_pos - data.site_xpos[site_id]
        # Note target_pos is where we want to be with this new action
        print("Remaining error in reaching target pos:", np.round(error, 5))

        current_pos = data.site_xpos[site_id].copy()
        distance = np.linalg.norm(cube_pos - current_pos)
        print("Distance from cube: ", distance)
        if distance < 0.02:
            print("Closong the gripper")
            close_gripper(model, data, viewer)

        print(
            f"Iteration {iteration}: "
            f"render={t1-t0:.3f}s | "
            f"VLA={t2-t1:.3f}s | "
            f"setup={t3-t2:.3f}s | "
            f"IK={t4-t3:.3f}s | "
            f"TOTAL={t4-t0:.3f}s"
        )
    print("👋 Simulation complete. Rendering video inline...")
    media.show_video(frames, fps=15)
    print("Gooooooooooooooooood!!!!!")


if __name__ == "__main__":
    run_simulation()

MuJoCo: 3.3.1
MUJOCO_GL: egl
SciPy      : 1.16.3 /usr/local/lib/python3.12/dist-packages/scipy/__init__.py
Mediapy    : /content/drive/MyDrive/colab_env_openvla/lib/python3.12/site-packages/mediapy/__init__.py
PyTorch Version: 2.10.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Transformers: 4.40.1
